In [ ]:
# V5 prerequisite check; no fallback to historical outputs.
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
V5_ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),Path('/content/drive/MyDrive/Musanga/Final/V5')] if (p.parent/'Data').is_dir()),None)
assert V5_ROOT is not None, 'Permanent Final/Data not found'
required=["05_rgb_dsm_ms_matched_v1/05_harmonized_sensor_registry.csv","05_rgb_dsm_ms_matched_v1/06_all_five_models_matched_unit_metrics.csv","05_rgb_dsm_ms_matched_v1/06_reporting_contract.json","05_rgb_dsm_ms_matched_v1/06_all_106_unit_support.csv","04_unified_rgb_heldout_v5/04_heldout_unit_metrics.csv"]
missing=[str(V5_ROOT/'Outputs'/p) for p in required if not (V5_ROOT/'Outputs'/p).is_file()]
assert not missing, 'Missing native V5 prerequisites: '+str(missing)



# Musanga V5 — NB05bis | UAV acquisition quality versus held-out model error

**Purpose.** Test whether poorer MOS UAV imagery contributes to Musanga prediction errors, separately from observed Musanga abundance and site differences.

**No GPU / no retraining.** Read the native V5 GIS footprints, original RGB mosaics, audited 4-band same-flight MS rasters, and frozen NB05 five-model held-out scores. Compute interpretable image-quality proxies on bounded 2 m / 0.5 m footprint excerpts, save QA maps/tables, and test whether quality predicts the error *within MOS* and across forests.

**Cautions:** UAV flight speed, camera settings, true ground sampling resolution, image overlap, SfM reprojection error, dates and calibration quality require acquisition metadata and photogrammetry reports. The proxies derived from orthomosaics are **not** proof of those properties. Low gradient energy may reflect genuinely homogeneous canopy rather than bad image. No prediction errors may be used to tune thresholds or decide which observations to exclude from the primary NB05 evaluation. Images are sampled at equal **physical** spatial scales, independent of native RGB pixel sizes.

**Outputs:** per-footprint indicators (n=106 including 14 MS-unmatched), per-flight MS coverage, example imagery and quality maps, paired prediction-error relationships on matched footprints (92 in the completed V5 run), within-MOS checks controlling for observed Musanga cover, optional spatial-group bootstrap, concise interpretation.

**First run:** use SITE_FILTER=None and SAMPLE_M=0.50; bounded windows and no full-mosaic reads. Optionally set SITE_FILTER='MOS' during troubleshooting, but full analysis needs all sites.

## 00 — Read-only sources and audit parameters

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely scipy pandas matplotlib scikit-learn statsmodels
from pathlib import Path
from contextlib import ExitStack
from collections import defaultdict
import json,math,os,re,warnings
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.transform import Affine
from rasterio.features import rasterize
from rasterio.windows import Window
from scipy.ndimage import gaussian_filter, sobel,laplace, distance_transform_edt
from scipy.stats import spearmanr
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in (Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),
                        Path('/content/drive/MyDrive/Musanga/Final/V5')) if p.is_dir()),None)
assert ROOT is not None
DATA=ROOT.parent/'Data'; V4=ROOT/'Outputs'
NB05=V4/'05_rgb_dsm_ms_matched_v1'
SPLIT=V4/'01_spatial_cover_folds_v2'
NB02=V4/'02_rgb_circles_plus_plots_v3'
OUT=V4/'05bis_acquisition_quality_v1';FIG=OUT/'figures'
for p in (OUT,FIG):p.mkdir(parents=True,exist_ok=True)
V31=json.loads((ROOT/'Outputs/00_input_audit/00_input_manifest.json').read_text())
UNITS=gpd.read_file(SPLIT/'01_reference_groups_and_folds_v2.gpkg')
UNITS['unit_id']=UNITS.unit_id.astype(str)
OWNERS=pd.read_csv(NB02/'02_reference_mosaic_owners_v2.csv')
OWNERS['unit_id']=OWNERS.unit_id.astype(str)
UNITS=UNITS.merge(OWNERS[['unit_type','unit_id','mosaic','roi_mosaic_coverage_fraction']],
                  on=['unit_type','unit_id'],validate='one_to_one')
UNITS=UNITS[UNITS.roi_mosaic_coverage_fraction>=.95].copy()
assert len(UNITS)==106
REG=pd.read_csv(NB05/'05_harmonized_sensor_registry.csv')
assert {'group','band','data_band','path'}.issubset(REG.columns)
MOSAICS={}
for entry in V31['mosaics']:
 if str(entry['file']).endswith('.tif'):
  f=Path(entry['path'])
  if not f.is_file():f=DATA/'Ortho_RGB'/entry['file']
  assert f.is_file(),f
  MOSAICS[entry['file']]=f
assert len(MOSAICS)==6
# Use NB05 exact registry; never mix individual bands from different flights.
GROUP_MOSAIC={
 'Baego_MS':'Baego.tif',
 'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
 'MOS_3_4_MS':'Mos_3_4_transparent_mosaic_group1_georef.tif',
 'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
 'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
 'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
for group in REG.group.unique():
 if str(group).startswith('Yoko_Carlier2020_MicaSense_'):
  choices=[m for m in MOSAICS if 'yoko' in m.lower()]
  assert len(choices)==1,choices
  GROUP_MOSAIC[group]=choices[0]
assert set(REG.group)==set(GROUP_MOSAIC),set(REG.group)-set(GROUP_MOSAIC)
def bands_for(group):
 q=REG[REG.group.eq(group)]
 assert len(q)==4 and set(q.band)=={'red','green','red_edge','nir'},group
 return {band:(Path(q[q.band.eq(band)].iloc[0].path),
                int(q[q.band.eq(band)].iloc[0].data_band))
         for band in ('red','green','red_edge','nir')}
SITE_FILTER=None # 'MOS' for fast troubleshooting; None for full site comparisons
SAMPLE_M=.50
MAX_DIM=800 # cap per-footprint window on 0.5m grid
BUFFER_M=4
MIN_PIXELS=100
LOCAL_DIAGNOSTIC_M=4
assert SITE_FILTER is None or SITE_FILTER in UNITS.site.unique()
if SITE_FILTER: UNITS=UNITS[UNITS.site.eq(SITE_FILTER)].copy()
print('Reference footprints:',len(UNITS),'| RGB mosaics:',len(MOSAICS),
      '| four-band MS acquisitions:',len(GROUP_MOSAIC),'| output:',OUT)


## 01 — Native source properties and acquisition metadata inventory

Resolutions, imagery channel types and NoData are *measured from raster headers*. Flight speed, exposure, camera parameters, image overlap and acquisition dates remain **not available** unless source metadata actually supply them. A clearly marked manual metadata template is written for optional completion later.

In [ ]:
rows=[]
for name,path in MOSAICS.items():
 with rasterio.open(path) as src:
  rows.append(dict(kind='RGB',mosaic=name,group='',band='RGB',
      path=str(path),width=src.width,height=src.height,
      native_pixel_x_m=abs(src.transform.a),native_pixel_y_m=abs(src.transform.e),
      channels=src.count,dtype=str(src.dtypes[0]),nodata=str(src.nodata),
      crs=str(src.crs),date_tag=str({k:v for k,v in src.tags().items()
                                  if any(w in k.lower() for w in ('date','time','flight'))})[:400]))
for group in GROUP_MOSAIC:
 for band,(path,index) in bands_for(group).items():
  with rasterio.open(path) as src:
   rows.append(dict(kind='MS',mosaic=GROUP_MOSAIC[group],
      group=group,band=band,path=str(path),width=src.width,height=src.height,
      native_pixel_x_m=abs(src.transform.a),native_pixel_y_m=abs(src.transform.e),
      channels=src.count,dtype=str(src.dtypes[index-1]),nodata=str(src.nodata),
      crs=str(src.crs),date_tag=str({k:v for k,v in src.tags().items()
                                  if any(w in k.lower() for w in ('date','time','flight'))})[:400]))
SOURCES=pd.DataFrame(rows);SOURCES.to_csv(OUT/'01_source_raster_properties.csv',index=False)
display(SOURCES[['kind','mosaic','group','band','native_pixel_x_m',
                 'native_pixel_y_m','date_tag']])
acq_cols=['mosaic','group','acquisition_date','flight_speed_m_s','altitude_m',
          'exposure_s','forward_overlap_pct','side_overlap_pct',
          'tiepoint_density','reprojection_RMSE_px','processing_report_path',
          'source_note']
template=SOURCES[['mosaic','group']].drop_duplicates().copy()
for key in acq_cols[2:]:template[key]=pd.NA
metadata_file=OUT/'01_optional_acquisition_metadata_template.csv'
if not metadata_file.is_file():
 template.to_csv(metadata_file,index=False)
print('Optional metadata template:',metadata_file)


## 02 — Derive footprint-specific RGB/MS quality proxies (bounded raster reads only)

Proxies are computed on the same fixed **0.5 m physical grid** within each reference footprint, using one complete MS flight. RGB measures use luminance contrast, spatial gradient energy and Laplacian high-frequency energy. MS measures include valid area, spectral saturation/contrast, local spectral discontinuity, RGB–MS gradient correspondence and registration-shift proxy. These are **relative indicators**, not direct measures of GSD, SfM quality or motion blur.

Registration proxy uses correlation of **gradient magnitudes**, not colour intensity; vegetation spectral changes and unequal native resolution may bias it, so the maximum shift is *diagnostic only*. Masks and imagery outside footprint are excluded from summaries. Samples are capped at MAX_DIM per side.

In [ ]:
def physical_grid(geom,crs):
 xmin,ymin,xmax,ymax=geom.bounds
 pad=BUFFER_M
 width=xmax-xmin+2*pad;height=ymax-ymin+2*pad
 pixel=max(SAMPLE_M,width/MAX_DIM,height/MAX_DIM)
 cols=max(1,int(np.ceil(width/pixel)));rows=max(1,int(np.ceil(height/pixel)))
 tr=Affine(pixel,0,xmin-pad,0,-pixel,ymax+pad)
 mask=rasterize([(geom,1)],out_shape=(rows,cols),transform=tr,
                 fill=0,dtype='uint8').astype(bool)
 return (rows,cols),tr,mask,pixel
def aligned_read(path,index,shape,tr,crs):
 with rasterio.open(path) as src:
  with WarpedVRT(src,crs=crs,transform=tr,
                 width=shape[1],height=shape[0],
                 resampling=Resampling.bilinear,nodata=np.nan) as vrt:
   a=vrt.read(index,masked=True,out_dtype='float32')
   v=np.asarray(a.filled(np.nan),dtype='float32')
   valid=(~np.ma.getmaskarray(a))&np.isfinite(v)
 return v,valid
def norm_quantile(x,mask,low=2,high=98):
 good=mask&np.isfinite(x)
 if good.sum()<MIN_PIXELS:return np.zeros_like(x,np.float32)
 lo,hi=np.percentile(x[good],[low,high])
 return np.clip((x-lo)/max(hi-lo,1e-6),0,1).astype('float32')
def luminance(rgb):
 return rgb[0]*.2126+rgb[1]*.7152+rgb[2]*.0722
def image_edges(x,sigma=1):
 smooth=gaussian_filter(np.nan_to_num(x),sigma)
 return np.hypot(sobel(smooth,axis=0),sobel(smooth,axis=1))
def trimmed_std(x,mask):
 v=x[mask&np.isfinite(x)]
 if len(v)<MIN_PIXELS:return np.nan
 lo,hi=np.percentile(v,[2,98]);v=v[(v>=lo)&(v<=hi)]
 return float(np.std(v)) if len(v) else np.nan
def score_footprint(z):
 mosaic=str(z.mosaic)
 with rasterio.open(MOSAICS[mosaic]) as src:
  geom=gpd.GeoSeries([z.geometry],crs=UNITS.crs).to_crs(src.crs).iloc[0]
  shape,tr,inside,pixel=physical_grid(geom,src.crs)
  rgb=[];rvalid=[]
  for k in (1,2,3):
   a,v=aligned_read(MOSAICS[mosaic],k,shape,tr,src.crs)
   rgb.append(a);rvalid.append(v)
  rgb=np.stack(rgb)
  rgbvalid=np.logical_and.reduce(rvalid)&inside
  gray=luminance(rgb)
  norm=norm_quantile(gray,rgbvalid)
  edges=image_edges(norm,1.)
  # Restrict sharpness to genuine boundaries to distinguish blur from
  # uniform canopy. Still only a scene-dependent proxy.
  edge_cut=np.percentile(edges[rgbvalid],70) if rgbvalid.sum()>MIN_PIXELS else np.inf
  strong=(edges>=edge_cut)&rgbvalid
  lap=np.abs(laplace(gaussian_filter(norm,0.65)))
  gradient=float(np.median(edges[strong])) if strong.any() else np.nan
  sharp=float(np.median(lap[strong])) if strong.any() else np.nan
  contrast=trimmed_std(gray,rgbvalid)
  sat=((rgb<=1).any(axis=0)|(rgb>=254).any(axis=0))&rgbvalid
  def base(group):
   layers=[];good=[]
   for band,(path,index) in bands_for(group).items():
    a,v=aligned_read(path,index,shape,tr,src.crs)
    layers.append(a);good.append(v&(a>0))
   a=np.stack(layers);ok=np.logical_and.reduce(good)&rgbvalid
   n=int(ok.sum())
   result=dict(ms_group=group,ms_valid_fraction=float(n/max(inside.sum(),1)),
      ms_rgb_common_fraction=float(n/max(rgbvalid.sum(),1)))
   if n<MIN_PIXELS:return result,None
   rel=np.where(ok[None],a/np.maximum(np.nansum(a,axis=0,keepdims=True),1e-8),0)
   nir,red,rededge=rel[3],rel[0],rel[2]
   ndvi=(a[3]-a[0])/np.maximum(a[3]+a[0],1e-8)
   result['ms_ndvi_iqr']=float(np.percentile(ndvi[ok],75)-np.percentile(ndvi[ok],25))
   result['ms_spectral_heterogeneity']=float(np.nanmedian(
       np.abs(ndvi[ok]-np.nanmedian(ndvi[ok]))))
   # A major RGB–MS geometry offset can be diagnosed by seeking correlation
   # of spatial edge magnitudes within this tiny reference footprint.
   # Not an accurate co-registration estimator in texture-poor scenes.
   lum=norm_quantile(gray,ok)
   spectral=norm_quantile(ndvi,ok)
   e1=image_edges(lum);e2=image_edges(spectral)
   finite=ok&np.isfinite(e1)&np.isfinite(e2)
   try:
    from scipy.ndimage import shift
    shifts=[(-2,0),(2,0),(0,-2),(0,2),(0,0),(-2,-2),(2,2)]
    tests=[]
    for dy,dx in shifts:
     z2=shift(e2,(dy,dx),order=0,mode='constant',cval=0)
     valid=finite&(shift(finite.astype('float32'),(dy,dx),order=0)>0)
     if valid.sum()<MIN_PIXELS:continue
     aa=e1[valid];bb=z2[valid]
     if aa.std()<1e-6 or bb.std()<1e-6:continue
     tests.append((float(np.corrcoef(aa,bb)[0,1]),dy,dx))
    if tests:
     win=max(tests,key=lambda x:x[0])
     zero=[x[0] for x in tests if x[1:]==(0,0)]
     result['rgb_ms_edge_corr0']=float(zero[0]) if zero else np.nan
     result['rgb_ms_best_edge_corr']=win[0]
     result['rgb_ms_shift_proxy_m']=float(pixel*np.hypot(win[1],win[2]))
   except Exception:pass
   result['ms_ndvi_gradient']=float(np.median(image_edges(
       norm_quantile(ndvi,ok))[ok]))
   return result,(a,ok,ndvi)
  candidates=[]
  for group,owner in GROUP_MOSAIC.items():
   if owner==mosaic:candidates.append(base(group))
  if not candidates:raise ValueError('No registered MS acquisition: '+mosaic)
  # Match NB05 operational selection: choose one complete acquisition
  # with greatest four-band support; never combine bands across flights.
  chosen=max(candidates,key=lambda x:x[0]['ms_valid_fraction'])
  result=chosen[0]
  result.update(site=z.site,unit_type=z.unit_type,unit_id=str(z.unit_id),
      mosaic=mosaic,fold=int(z.fold),
      rgb_valid_fraction=float(rgbvalid.sum()/max(inside.sum(),1)),
      rgb_gradient_proxy=gradient,rgb_laplacian_edge_proxy=sharp,
      rgb_local_contrast=contrast,
      rgb_saturation_fraction=float(sat.sum()/max(rgbvalid.sum(),1)),
      sample_pixel_m=float(pixel),
      reference_pixels=int(inside.sum()),
      rgb_native_gsd_m=float(abs(src.transform.a)),
      observed_cover_source='NB05 common valid (used only after merge)')
  return result
QUALITY=[]
for i,z in enumerate(UNITS.itertuples(index=False)):
 try:QUALITY.append(score_footprint(z))
 except Exception as exc:
  QUALITY.append(dict(site=z.site,unit_type=z.unit_type,unit_id=str(z.unit_id),
                  mosaic=z.mosaic,fold=z.fold,qa_error=str(exc)[:300]))
 if (i+1)%20==0:print('Assessed',i+1,'of',len(UNITS),flush=True)
QUALITY=pd.DataFrame(QUALITY)
assert len(QUALITY)==len(UNITS)
QUALITY.to_csv(OUT/'02_reference_image_quality_metrics.csv',index=False)
print('Quality extraction completed; missing or failed footprints retained as NA.')
display(QUALITY.groupby('site').agg(
 n=('unit_id','size'),rgb_GSD_m=('rgb_native_gsd_m','median'),
 sharpness_proxy=('rgb_laplacian_edge_proxy','median'),
 ms_coverage=('ms_valid_fraction','median')).round(3))


## 03 — Join NB05 five-arm held-out errors and create visual diagnostics

**No selection on errors.** This is an explanatory observational analysis, not a new held-out score. The existing 92-unit support is used for paired comparisons. Raw RGB-only 106-unit quality measurements remain separately available.

In [ ]:
PRED_PATH=NB05/'06_all_five_models_matched_unit_metrics.csv'
assert PRED_PATH.is_file(),'Run NB05 Section 06 reporting first'
PRED=pd.read_csv(PRED_PATH)
PRED['unit_id']=PRED.unit_id.astype(str)
REPORT_CONTRACT=json.loads((NB05/'06_reporting_contract.json').read_text())
SUPPORT=pd.read_csv(NB05/'06_all_106_unit_support.csv',dtype={'unit_id':str})
assert len(SUPPORT)==106 and not SUPPORT.duplicated(['unit_type','unit_id']).any()
EXPECTED={'RGB','RGB control','RGB+DSM','RGB+MS','RGB+DSM+MS'}
matched=SUPPORT[SUPPORT.n_common.gt(0)]
assert len(matched)==REPORT_CONTRACT['matched_units']
assert set(PRED.model)==EXPECTED
assert len(PRED)==len(matched)*len(EXPECTED)
assert not PRED.duplicated(['unit_type','unit_id','model']).any()
assert PRED.groupby(['unit_type','unit_id']).model.nunique().eq(len(EXPECTED)).all()
assert set(zip(PRED.unit_type,PRED.unit_id))==set(zip(matched.unit_type,matched.unit_id))
assert PRED.groupby(['unit_type','unit_id']).n_pixels.nunique().eq(1).all()
assert PRED.groupby(['unit_type','unit_id']).observed_cover_pp.nunique().eq(1).all()
if SITE_FILTER: PRED=PRED[PRED.site.eq(SITE_FILTER)].copy()
print('Native V5 predictions:',len(PRED),'rows; models:',sorted(PRED.model.unique()))
JOIN=PRED.merge(QUALITY.drop(columns=['fold','site','mosaic'],errors='ignore'),
            on=['unit_type','unit_id'],validate='many_to_one')
assert len(JOIN)==len(PRED)
JOIN['signed_error_pp']=JOIN.predicted_cover_pp-JOIN.observed_cover_pp
JOIN['absolute_error_pp']=JOIN.signed_error_pp.abs()
JOIN.to_csv(OUT/'03_quality_joined_five_model_errors.csv',index=False)
# Full RGB-only diagnostics remain independent of the joint sensor coverage.
FULL_RGB=pd.read_csv(ROOT/'Outputs/04_unified_rgb_heldout_v5/04_heldout_unit_metrics.csv',dtype={'unit_id':str})
assert len(FULL_RGB)==106 and FULL_RGB.model.eq('unified').all()
if SITE_FILTER: FULL_RGB=FULL_RGB[FULL_RGB.site.eq(SITE_FILTER)].copy()
FULL_JOIN=FULL_RGB.merge(QUALITY.drop(columns=['fold','site','mosaic'],errors='ignore'),on=['unit_type','unit_id'],validate='one_to_one')
assert len(FULL_JOIN)==len(FULL_RGB)
FULL_JOIN['absolute_error_pp']=(FULL_JOIN.predicted_cover_pp-FULL_JOIN.observed_cover_pp).abs()
FULL_JOIN.to_csv(OUT/'03_full_RGB_106_quality_errors.csv',index=False)
KEYS=['rgb_native_gsd_m','rgb_gradient_proxy','rgb_laplacian_edge_proxy',
      'rgb_local_contrast','rgb_saturation_fraction',
      'ms_valid_fraction','ms_ndvi_iqr','ms_ndvi_gradient',
      'rgb_ms_edge_corr0','rgb_ms_shift_proxy_m']
for key in KEYS:
 if key not in JOIN:JOIN[key]=np.nan
p=JOIN[JOIN.model.eq('RGB+MS')].copy()
assert len(p)==len(PRED)//len(EXPECTED)
fig,axs=plt.subplots(2,3,figsize=(14,8),layout='constrained')
for ax,key in zip(axs.flat,[
    'rgb_native_gsd_m','rgb_laplacian_edge_proxy','rgb_local_contrast',
    'ms_valid_fraction','rgb_ms_edge_corr0','rgb_ms_shift_proxy_m']):
 for site,q in p.groupby('site'):
  valid=q[[key,'absolute_error_pp']].replace([np.inf,-np.inf],np.nan).dropna()
  ax.scatter(valid[key],valid.absolute_error_pp,alpha=.7,s=25,label=site)
 ax.set(xlabel=key,ylabel='Absolute RGB+MS cover error (pp)')
 ax.grid(alpha=.15)
axs.flat[0].legend(frameon=False)
fig.suptitle('UAV quality indicators and held-out RGB+MS error (descriptive)')
fig.savefig(FIG/'03_quality_vs_error_scatter.png',dpi=175,bbox_inches='tight')
plt.show()
corr=[]
for site,subset in [('ALL',p),*list(p.groupby('site'))]:
 for key in KEYS:
  q=subset[[key,'absolute_error_pp','observed_cover_pp']].replace(
      [np.inf,-np.inf],np.nan).dropna()
  if len(q)<6 or q[key].nunique()<4:continue
  rho,pval=spearmanr(q[key],q.absolute_error_pp)
  corr.append(dict(site=site,quality_metric=key,n=len(q),
                   rho_spearman=float(rho),p_nominal=float(pval)))
CORR=pd.DataFrame(corr)
CORR.to_csv(OUT/'03_quality_error_associations_exploratory.csv',index=False)
print('Descriptive associations; nominal p values do not correct multiple testing.')
display(CORR[CORR.site.eq('MOS')].round(3))


## 04 — Control for reference Musanga cover and assess MOS specifically

Small samples and correlated quality metrics limit statistical power. For each metric, report **partial Spearman correlation** between rank-transformed quality and rank-transformed absolute error after residualizing against rank-transformed observed cover. This is a diagnostic, not causal inference. Also show whether model improvements (RGB+MS minus matched fine-tuned RGB control absolute error) covary with quality.

In [ ]:
from scipy.stats import rankdata
from sklearn.linear_model import LinearRegression
def partial_rho(frame,predictor,response='absolute_error_pp',control='observed_cover_pp'):
 q=frame[[predictor,response,control]].replace(
    [np.inf,-np.inf],np.nan).dropna()
 if len(q)<8 or q[predictor].nunique()<4:return np.nan,len(q)
 y=rankdata(q[response]);x=rankdata(q[predictor])
 c=rankdata(q[control]).reshape(-1,1)
 lm=LinearRegression()
 xr=x-lm.fit(c,x).predict(c)
 yr=y-lm.fit(c,y).predict(c)
 if np.std(xr)<1e-8 or np.std(yr)<1e-8:return np.nan,len(q)
 return float(np.corrcoef(xr,yr)[0,1]),len(q)
part=[]
for site,sub in [('ALL',p),*list(p.groupby('site'))]:
 for k in KEYS:
  v,n=partial_rho(sub,k)
  part.append(dict(site=site,quality_metric=k,
                   partial_spearman_given_observed_cover=v,n=n))
PARTIAL=pd.DataFrame(part)
PARTIAL.to_csv(OUT/'04_partial_quality_error_correlations.csv',index=False)
full_part=[]
for site,sub in [('ALL',FULL_JOIN),*list(FULL_JOIN.groupby('site'))]:
 for k in KEYS:
  if k not in sub:continue
  v,n=partial_rho(sub,k)
  full_part.append(dict(site=site,quality_metric=k,partial_spearman_given_observed_cover=v,n=n))
pd.DataFrame(full_part).to_csv(OUT/'04_full_RGB_partial_quality_error_correlations.csv',index=False)
print('MOS cover-adjusted quality-error relationship:')
display(PARTIAL[PARTIAL.site.eq('MOS')].round(3))
pair=JOIN.pivot_table(index=['site','unit_type','unit_id','spatial_group',
   'observed_cover_pp'],columns='model',values='absolute_error_pp').reset_index()
pair['delta_MS_vs_RGB']=pair['RGB+MS']-pair['RGB control']
pair['delta_MS_vs_frozen_RGB']=pair['RGB+MS']-pair['RGB']
qp=pair.merge(QUALITY,on=['unit_type','unit_id'],validate='one_to_one',
               suffixes=('','_qa'))
qp.to_csv(OUT/'04_paired_MS_RGB_quality.csv',index=False)
fig,axes=plt.subplots(1,3,figsize=(14,4),layout='constrained')
for ax,key in zip(axes,['rgb_laplacian_edge_proxy','ms_valid_fraction',
                        'rgb_ms_edge_corr0']):
 for site,grp in qp.groupby('site'):
  ax.scatter(grp[key],grp.delta_MS_vs_RGB,alpha=.65,s=25,label=site)
 ax.axhline(0,color='0.35',linestyle='--',lw=1)
 ax.set(xlabel=key,ylabel='Δ absolute error: RGB+MS − RGB control (pp)')
 ax.grid(alpha=.12)
axes[0].legend(frameon=False)
fig.suptitle('Does MS help relative to matched RGB control where image quality is poor?')
fig.savefig(FIG/'04_MS_gain_vs_quality.png',dpi=170,bbox_inches='tight')
plt.show()


## 05 — Representative image panels: compare RGB detail, MS spectra and reference boundaries

Select sites by footprint cover quantiles, **not by held-out error**. Display RGB, edge strength, MS false colour and 4-band validity for the same spatial footprint. These plots allow a reviewer to judge whether low detail is genuine or reflects blur, seams or mosaicking. A low-quality label must not be assigned automatically on this proxy alone.

In [ ]:
figrows=[]
for site,grp in QUALITY.groupby('site'):
 subset=UNITS[UNITS.site.eq(site)]
 if not len(subset):continue
 q=grp.sort_values('rgb_laplacian_edge_proxy',na_position='last')
 choices=q.iloc[sorted(set([0,len(q)//2,len(q)-1]))]
 for item in choices.itertuples(index=False):
  candidate=subset[(subset.unit_type.eq(item.unit_type))&
                   (subset.unit_id.eq(str(item.unit_id)))]
  if len(candidate)==0:continue
  z=candidate.iloc[0]
  with rasterio.open(MOSAICS[z.mosaic]) as src:
   geom=gpd.GeoSeries([z.geometry],crs=UNITS.crs).to_crs(src.crs).iloc[0]
   shape,tr,inside,pixel=physical_grid(geom,src.crs)
   rgbarr=np.stack([aligned_read(MOSAICS[z.mosaic],i,shape,tr,src.crs)[0]
                    for i in (1,2,3)])
   rgbvalid=np.isfinite(rgbarr).all(axis=0)&inside
   rgb_display=np.moveaxis(np.clip(rgbarr/255.,0,1),0,-1)
   gray=luminance(rgbarr)
   detail=image_edges(norm_quantile(gray,rgbvalid))
   group=item.ms_group if isinstance(item.ms_group,str) else None
   if group not in GROUP_MOSAIC:continue
   ms=[];valid=[]
   for b,(path,index) in bands_for(group).items():
    a,v=aligned_read(path,index,shape,tr,src.crs)
    ms.append(a);valid.append(v&(a>0))
   ms=np.stack(ms);good=np.logical_and.reduce(valid)&inside
   false=np.zeros((shape[0],shape[1],3),float)
   for j,band in enumerate([3,2,0]):
    false[:,:,j]=norm_quantile(ms[band],good)
   false[~good]=1.
   fig,ax=plt.subplots(1,4,figsize=(13,3.4),layout='constrained')
   ax[0].imshow(rgb_display)
   ax[1].imshow(detail,cmap='magma')
   ax[2].imshow(false)
   ax[3].imshow(good,cmap='gray',vmin=0,vmax=1)
   for axis,title in zip(ax,['RGB','RGB detail proxy','MS false colour','MS coverage']):
    axis.contour(inside,levels=[.5],colors=['#ffe24a'],linewidths=.8)
    axis.set_title(title);axis.axis('off')
   fig.suptitle(f'{site} {z.unit_type} {z.unit_id} | {group}')
   path=FIG/f'05_{site}_{z.unit_type}_{z.unit_id}.png'
   fig.savefig(path,dpi=150,bbox_inches='tight');plt.show();plt.close(fig)
   figrows.append(dict(site=site,unit_type=z.unit_type,unit_id=z.unit_id,
                       image=str(path)))
IMAGE_INDEX=pd.DataFrame(figrows)
IMAGE_INDEX.to_csv(OUT/'05_visual_QA_examples.csv',index=False)
print('Figures saved:',len(IMAGE_INDEX))


## 06 — Executive interpretation and limitations

Interpret this notebook primarily through (i) descriptive site distributions of native GSD, edge sharpness, MS validity and RGB–MS gradient correspondence; (ii) within-MOS quality versus held-out error conditional on observed cover; and (iii) whether RGB+MS gains are more pronounced where RGB image quality is poor.

**Do not conclude:** that lower quality caused the observed error, that a scene-dependent texture proxy estimates true blur, or that the best registration proxy is a measured geolocation offset. Flight metadata and/or photogrammetric QA are needed to test those claims directly.

**Next decision:** only if strong and spatially consistent associations emerge, design quality-aware data augmentation or quality-weighted training; preserve the frozen NB05 outer folds. No change to existing models or predictions is made here.

In [ ]:
manifest=dict(notebook='NB05bis_v5_unified',n_footprints=len(QUALITY),
  n_matched=(len(p) if 'p' in globals() else None),
  indicators=KEYS,site_filter=SITE_FILTER,sample_m=SAMPLE_M,
  source_nb05=str(PRED_PATH),models=sorted(EXPECTED),primary_MS_comparator='RGB control',status='diagnostic_proxies_not_photogrammetry_QA')
(OUT/'05bis_manifest.json').write_text(json.dumps(manifest,indent=2))
print('NB05bis ready. Output:',OUT)
display(pd.DataFrame([dict(file=q.name,size_MB=round(q.stat().st_size/1e6,2))
 for q in sorted(OUT.glob('*.csv'))]))
